# Cross-gradient coupling as a prior

**Level 1 — Foundations**  
**Prerequisites:** Python basics; 35 min.

## Objectives

Trace the evidence from source contract to result. State units and assumptions, reproduce the calculation, and identify which inference the evidence cannot support.

## Setup

Uses repository modules and committed records through a discovered root; it downloads and mutates nothing. Install exact pins: `pip install numpy==2.2.6 matplotlib==3.11.2`

## Data

Both 3D property models are synthetic, with coordinates and method metrics stored in the case receipts.

## Method

Gravity density is g/cm³, magnetic susceptibility dimensionless, gravity data mGal. A shared-boundary prior is often represented by cross-gradient `g=∇ρ×∇κ`; penalizing `||g||²` encourages parallel gradients without forcing equal amplitudes. This is a structural assumption, not a physical law, and can hurt when property boundaries differ. `JOINT_SHARED` is a seeded synthetic 16×24×28 3D case with 80/80/70 m spacing, using SimPEG 0.25.2 integral equations and SciPy. Plot the committed truth, independent model and joint model on the physical x-z slice nearest the stored middle y-coordinate; use stored centers, not assumed reshaping. It is a replay of saved arrays, not a new inversion. Distinguish density units and the synthetic provenance.

The data operators have distinct units, coverage and null spaces. Cross-gradient penalties depend on discretization and scaling; parallel gradients do not imply equal amplitudes or shared geology everywhere. Compare predictions on each survey as well as truth error. Inspect units and masks before interpreting small ratio changes.

## Results

Read the physical axes and units; distinguish data fit, error against synthetic truth and agreement with analyst labels. Recomputed values must agree with the serialized receipt. A plotted model alone does not establish uniqueness, field transfer or resolution.

### Diagnostic 1

Inspect the committed case record and provenance.

In [ ]:
from pathlib import Path
from importlib.metadata import version
print("Runtime versions:", {name: version(name) for name in ['numpy', 'matplotlib']})
import numpy as np
root=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/"data").is_dir() and (p/"data-pipeline").is_dir())
# Load both seeded synthetic 3D artifacts and verify grid units/provenance.
import json
from pathlib import Path
cases={n:json.loads((root/f"data/derived/v2/{n}/regularization.json").read_text(encoding="utf-8")) for n in ("JOINT_SHARED","JOINT_CONFLICT")}
for n,d in cases.items(): print(n,"synthetic",d["provenance"]["synthetic"],"shape",d["grid"]["shape"],"spacing m",d["grid"]["spacing"],"density",d["units"],"engine",d["engine"])

### Diagnostic 2

Compute the declared comparison and plot it with physical units.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
# Plot truth and saved independent/joint density on the physical x-z slice at median y.
import matplotlib.pyplot as plt
d=cases["JOINT_SHARED"]; c=np.asarray(d["grid"]["centers"],float); y=np.median(np.unique(c[:,1])); s=np.isclose(c[:,1],y); vals=[np.asarray(d["truth"]),np.asarray(d["methods"]["joint-uncoupled"]["model"]),np.asarray(d["methods"]["joint"]["model"])]
fig,ax=plt.subplots(1,3,figsize=(11,3.5),constrained_layout=True)
for a,v,t in zip(ax,vals,["truth","independent","coupled"]): im=a.scatter(c[s,0],-c[s,2],c=v[s],s=28,cmap="viridis"); a.set(title=t,xlabel="Easting (m)",ylabel="Depth below datum (m)",aspect="equal"); fig.colorbar(im,ax=a,label="Density (g/cm³)")
plt.show(); print("slice northing m",y,"cells",s.sum())

## Summary

Conclusion is conditional on geometry, acquisition, noise model, split, parameterization and source rights. Preserve negative cases and name the next observation or control needed.

## References

- [https://docs.simpeg.xyz/latest/content/user-guide/tutorials/13_joint_inversion/plot_inv_3_cross_gradient_pf.html](https://docs.simpeg.xyz/latest/content/user-guide/tutorials/13_joint_inversion/plot_inv_3_cross_gradient_pf.html)
- [https://www.pygimli.org/user-guide/inversion/](https://www.pygimli.org/user-guide/inversion/)